<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_1_train_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.1. Entrenamiento de modelo Random Forest


## 0. Clonado de Repositorio, instalación de librería e importación.

### Clonado de Repositorio

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


### Acceso de Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Instalación de librerías

In [3]:
# !{sys.executable} -m pip install -q pandas_market_calendars  # Solo si usás horarios de mercados
!{sys.executable} -m pip install -q ta
print("Librerías instaladas: ta")

/bin/bash: line 1: {sys.executable}: command not found
Librerías instaladas: ta


### Importación de librerías

In [4]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Utilidades del sistema y fechas
import os
import glob
import requests
from datetime import datetime, timedelta

# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

# Análisis técnico
#import ta
#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

# Estadística
from scipy.stats import spearmanr

#que es?
#from tqdm.notebook import tqdm
from tqdm import tqdm
# Modelos ML
#from xgboost import XGBRegressor
#from sklearn.metrics import mean_squared_error, r2_score

# Calendario de mercados (descomentar si lo necesitás)
# import pandas_market_calendars as mcal

### Carga de mnq_model

In [5]:
def load_data_from_drive():

    # Definir la URL del archivo Parquet en Drive
    df_path_mnq = f'{drive_path}/mnq_data/mnq_model.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df_model = pd.read_parquet(df_path_mnq)

    return df_model

In [6]:
mnq_model = load_data_from_drive()

## 1. Carga de ventanas X_* escalad, y_* y el  escalador

In [7]:
import numpy as np
import joblib

def load_datasets_and_scaler(drive_path, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """
    folder = "ventanas_x_y_scaled" if scaled else "ventanas_x_y"
    scaled_name = "_scaled" if scaled else ""

    ruta_train  = f"{drive_path}/{folder}/mnq_Xy_train{scaled_name}.npz"
    ruta_valid  = f"{drive_path}/{folder}/mnq_Xy_valid{scaled_name}.npz"
    ruta_test   = f"{drive_path}/{folder}/mnq_Xy_test{scaled_name}.npz"


    ruta_scaler = f"{drive_path}/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(ruta_train)
    data_valid = np.load(ruta_valid)
    data_test  = np.load(ruta_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(ruta_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler

In [8]:
X_train_s, y_train, X_valid_s, y_valid, X_test_s, y_test, scaler = load_datasets_and_scaler(drive_path)

## 2. Selección y filtrado de features

In [9]:
 #Listado de features para el modelo
 features_random_forest = ['factor30', 'rsi_14', 'price_ema30', 'stoch_k_20', 'bb_percent_30_20', 'reversal_momentum_factor', 'rsi_7', 'reversal_media_factor', 'rsi_3', 'bb_percent_20_15']

In [10]:
    #Listado de features sin 'date'
    features =  mnq_model.columns.tolist()
    features.remove('date') #Remover fecha
    features.remove('target_return_30')  #Remover el target

Función para filtrar features de los X_*, siempre con los datos escalados

In [11]:
def filter_features_to_model(
    features_select,          #Listado de features a mantener
    X_train = X_train_s,    #Siempre usaremos los subsets escalados
    X_valid = X_valid_s,
    X_test = X_test_s,
    features = features,    #El listado de features siempre es el mismo.
    window_size = 60        #Siempre el mismo
        ):

    # Índices fijos de OHLCV
    idx_list = [0, 1, 2, 3, 4]

    for name in features_select:
        if name in features:
            idx_list.append(features.index(name))
        else:
            print(f"⚠️ Feature '{name}' no encontrado en features_list.")

    # Ordenar y eliminar duplicados
    idx_features_selected = sorted(set(idx_list))

    n_features = len(features)

    # Verificación rápida
    n_total_cols = window_size * n_features
    assert X_train.shape[1] == n_total_cols, "X_train no coincide con window_size * n_features"

    # Calcular columnas a mantener
    cols_to_keep = []
    for idx in idx_features_selected:
        start = idx * window_size
        end = (idx + 1) * window_size
        cols_to_keep.extend(range(start, end))

    # Filtrar arrays
    X_train_f = X_train[:, cols_to_keep]
    X_valid_f = X_valid[:, cols_to_keep]
    X_test_f  = X_test[:, cols_to_keep]

    # Features filtrados
    features_f = [features[i] for i in idx_features_selected]

    print(f"Features seleccionados ({len(features_f)}): {features_f}")
    print("X_train_f shape:", X_train_f.shape)
    print("X_valid_f shape:", X_valid_f.shape)
    print("X_test_f shape :", X_test_f.shape)

    return X_train_f, X_valid_f, X_test_f, features_f

In [12]:
X_train_model, X_valid_model, X_test_model, features_model = filter_features_to_model(
    features_select = features_random_forest,
)

Features seleccionados (15): ['open', 'high', 'low', 'close', 'volume', 'rsi_3', 'rsi_7', 'rsi_14', 'stoch_k_20', 'bb_percent_20_15', 'bb_percent_30_20', 'price_ema30', 'reversal_momentum_factor', 'reversal_media_factor', 'factor30']
X_train_f shape: (276017, 900)
X_valid_f shape: (59297, 900)
X_test_f shape : (59297, 900)


## 4. Entrenamiento de modelo

In [13]:
# --- IMPORTS & VERSION CHECK ---
import os, json, joblib, numpy as np, pandas as pd
import sklearn, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform

warnings.filterwarnings("ignore")

print("Python:", __import__("platform").python_version())
print("scikit-learn:", sklearn.__version__)  # -> debería ser 1.4.2

RANDOM_SEED = 42
rng = np.random.RandomState(RANDOM_SEED)

Python: 3.12.11
scikit-learn: 1.6.1
